# RAG Experiments

This notebook inspects the corpus, retrieval results, source-grounded answers and evaluation experiments. It imports reusable code from `src/rag_assistant` rather than duplicating the implementation.

## Environment

Install the project with the retrieval, generation and dev extras listed in the README. Prepare the raw corpus and extracted sections first. Start Ollama, select the project virtual environment and run the cells in order. The saved outputs come from a complete execution; generated answers still require manual review.

## 1. Corpus preparation

A RAG system needs an evidence corpus before it can answer questions. This project uses an explicit allowlist of official scikit-learn and pandas documentation pages. It does not crawl arbitrary web pages.

```text
Configured source URLs
        ↓
Downloaded raw HTML pages
        ↓
Provenance manifest with checksums
        ↓
Extracted heading-aware sections
        ↓
Chunks for retrieval (next step)
```

Raw HTML is the page content sent by a documentation website. It is stored locally so that extraction can be reproduced. Navigation, sidebars, scripts and other webpage elements are removed during extraction.

In [1]:
from collections import Counter
from pathlib import Path

from rag_assistant.config import load_sources

project_root = Path.cwd().resolve()
if not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
if not (project_root / 'pyproject.toml').exists():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')

sources = load_sources(project_root / 'config' / 'sources.toml')
print(f'Configured sources: {len(sources)}')
print(dict(Counter(source.library for source in sources)))
for source in sources[:5]:
    print(f'- {source.identifier}: {source.title}')

Configured sources: 25
{'scikit-learn': 17, 'pandas': 8}
- skl-pitfalls: Common pitfalls and recommended practices
- skl-compose: Pipelines and composite estimators
- skl-preprocessing: Preprocessing data
- skl-impute: Imputation of missing values
- skl-validation: Cross-validation: evaluating estimator performance


### Raw documents and provenance

`rag-assistant ingest` downloads every allowlisted page and saves the original HTML under `data/raw/`. It also creates `manifest.json`. The manifest records the source URL, retrieval time, content type, file size and SHA-256 checksum. A checksum is a fingerprint: when the downloaded page changes, its checksum changes too.

The raw files and manifest remain local and are excluded from Git.

In [2]:
import hashlib
import json

raw_directory = project_root / 'data' / 'raw'
manifest_path = raw_directory / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError('Run `rag-assistant ingest` from the repository root first.')

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
first_document = manifest['sources'][0]
print(f"Downloaded documents: {len(manifest['sources'])}")
print(f"Example document: {first_document['title']}")
print(f"Source URL: {first_document['retrieved_url']}")
print(f"Retrieved at: {first_document['retrieved_at_utc']}")
print(f"SHA-256: {first_document['sha256']}")

Downloaded documents: 25
Example document: Common pitfalls and recommended practices
Source URL: https://scikit-learn.org/stable/common_pitfalls.html
Retrieved at: 2026-09-30T09:57:46.268187+00:00
SHA-256: 21fac11c181ab35624dcf8673f0475d403d4f6ecf0f9bead01a573d7642b515c


### Extracted sections

`rag-assistant extract` reads the raw HTML and keeps the documentation article only. It preserves headings, paragraphs, list items and code blocks.

A **section** is a meaningful source unit under a heading. For example, a section may contain all the text under `Preprocessing data > Standardization`. A section is not yet a retrieval chunk: it can be too long for embedding and search. The next stage will split each section into smaller overlapping chunks while keeping its source metadata.

In [3]:
sections_path = project_root / 'data' / 'processed' / 'sections.jsonl'
if not sections_path.exists():
    raise FileNotFoundError('Run `rag-assistant extract` from the repository root first.')

sections = [json.loads(line) for line in sections_path.read_text(encoding='utf-8').splitlines()]
print(f'Extracted sections: {len(sections)}')
print(dict(Counter(section['library'] for section in sections)))

example_section = sections[1]
print(f"\nSection ID: {example_section['section_id']}")
print(f"Heading path: {' > '.join(example_section['section_path'])}")
print(f"Source: {example_section['source_url']}")
print('\nText preview:')
print(example_section['text'][:700])

Extracted sections: 645
{'scikit-learn': 337, 'pandas': 308}

Section ID: skl-pitfalls:002
Heading path: 12. Common pitfalls and recommended practices > 12.1. Inconsistent preprocessing
Source: https://scikit-learn.org/stable/common_pitfalls.html

Text preview:
scikit-learn provides a library of Dataset transformations , which
may clean (see Preprocessing data ), reduce
(see Unsupervised dimensionality reduction ), expand (see Kernel Approximation )
or generate (see Feature extraction ) feature representations.
If these data transforms are used when training a model, they also
must be used on subsequent datasets, whether it’s test data or
data in a production system. Otherwise, the feature space will change,
and the model will not be able to perform effectively.

For the following example, let’s create a synthetic dataset with a
single feature:

```
>>> from sklearn.datasets import make_regression
>>> from sklearn.model_selection import train_test_


### Validation and next step

The initial corpus contains 25 documents and 645 non-empty extracted sections. Every raw file is checked against the SHA-256 value in the manifest, and every configured document is represented by extracted sections.

Next, we will inspect baseline chunking before embeddings are generated.

## 2. Baseline chunking

A source section can be too long to embed and retrieve precisely. The baseline splits each section into windows of 250 whitespace-separated units with 40 units of overlap. The overlap repeats the end of one chunk at the beginning of the next chunk, which keeps context around a boundary.

Each chunk keeps its section ID, document ID, heading path and source URL. This is a simple reference strategy; later it will be compared with structure-aware and model-tokenizer-based chunking.

In [4]:
from rag_assistant.chunking import chunk_sections
from rag_assistant.config import load_chunking_settings

settings = load_chunking_settings(project_root / 'config' / 'settings.toml')
chunks_path = project_root / 'data' / 'processed' / 'chunks.jsonl'

def report_progress(completed, total, section_id):
    if completed == 1 or completed == total or completed % 25 == 0:
        print(f'Processed {completed}/{total} sections; latest: {section_id}')

chunks = chunk_sections(
    sections_path,
    chunks_path,
    settings,
    progress_callback=report_progress,
)

print(f'\nChunk size: {settings.chunk_size} units')
print(f'Chunk overlap: {settings.chunk_overlap} units')
print(f'Created chunks: {len(chunks)}')
print(f'Largest chunk: {max(chunk["unit_count"] for chunk in chunks)} units')

Processed 1/645 sections; latest: skl-pitfalls:001
Processed 25/645 sections; latest: skl-compose:011
Processed 50/645 sections; latest: skl-impute:003
Processed 75/645 sections; latest: skl-validation:018
Processed 100/645 sections; latest: skl-metrics:001
Processed 125/645 sections; latest: skl-metrics:026
Processed 150/645 sections; latest: skl-metrics:051
Processed 175/645 sections; latest: skl-linear-models:022
Processed 200/645 sections; latest: skl-neighbors:006
Processed 225/645 sections; latest: skl-trees:005
Processed 250/645 sections; latest: skl-ensembles:018
Processed 275/645 sections; latest: skl-clustering:005
Processed 300/645 sections; latest: skl-clustering:030
Processed 325/645 sections; latest: skl-feature-extraction:003
Processed 350/645 sections; latest: pd-missing:013
Processed 375/645 sections; latest: pd-groupby:002
Processed 400/645 sections; latest: pd-groupby:027
Processed 425/645 sections; latest: pd-indexing:011
Processed 450/645 sections; latest: pd-index

Processed 600/645 sections; latest: pd-reshaping:013
Processed 625/645 sections; latest: pd-cookbook:018
Processed 645/645 sections; latest: pd-cookbook:038



Chunk size: 250 units
Chunk overlap: 40 units
Created chunks: 1046
Largest chunk: 250 units


In [5]:
from rag_assistant.chunking import chunk_section

long_section = next(section for section in sections if len(section['text'].split()) > settings.chunk_size)
section_chunks = chunk_section(long_section, settings)

print(f"Section: {' > '.join(long_section['section_path'])}")
print(f"Section units: {len(long_section['text'].split())}")
print(f"Chunks created: {len(section_chunks)}")

for chunk in section_chunks[:2]:
    print(f"\n{chunk['chunk_id']} | units {chunk['unit_start']} to {chunk['unit_end']}")
    print(chunk['text'][:500])

Section: 12. Common pitfalls and recommended practices > 12.1. Inconsistent preprocessing
Section units: 278
Chunks created: 2

skl-pitfalls:002:chunk:001 | units 0 to 250
scikit-learn provides a library of Dataset transformations , which
may clean (see Preprocessing data ), reduce
(see Unsupervised dimensionality reduction ), expand (see Kernel Approximation )
or generate (see Feature extraction ) feature representations.
If these data transforms are used when training a model, they also
must be used on subsequent datasets, whether it’s test data or
data in a production system. Otherwise, the feature space will change,
and the model will not be able to perform ef

skl-pitfalls:002:chunk:002 | units 210 to 278
scaler.transform(X_test)
>>> mean_squared_error(y_test, model.predict(X_test_transformed))
0.90...
```

Alternatively, we recommend using a Pipeline , which makes it easier to chain transformations
with estimators, and reduces the possibility of forgetting a transformation:

```


## 3. Embeddings

An embedding model converts each chunk into a vector: a fixed-length list of numbers representing its meaning. Similar chunks should have vectors that are close together. The selected open-source model supports both English documentation and French questions.

The model requires `passage: ` before stored documentation chunks and `query: ` before future user questions. Vectors and their manifest are local artifacts and are excluded from Git.

In [6]:
from rag_assistant.config import load_embedding_settings
from rag_assistant.embeddings import create_embeddings

embedding_settings = load_embedding_settings(project_root / 'config' / 'settings.toml')
embedding_directory = project_root / 'artifacts' / 'embeddings'
embedding_file = embedding_directory / 'chunk_embeddings.npz'
manifest_path = embedding_directory / 'manifest.json'

current_chunks_sha256 = hashlib.sha256(chunks_path.read_bytes()).hexdigest()
existing_manifest = (
    json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest_path.exists()
    else {}
)
if embedding_file.exists() and existing_manifest.get('input_sha256') == current_chunks_sha256:
    embedding_manifest = existing_manifest
    print('Using existing local embeddings.')
else:
    embedding_manifest = create_embeddings(
        chunks_path,
        embedding_directory,
        embedding_settings,
        batch_size=32,
        show_progress_bar=True,
    )

print(embedding_manifest)

Using existing local embeddings.
{'model_name': 'intfloat/multilingual-e5-small', 'normalize_embeddings': True, 'document_prefix': 'passage: ', 'query_prefix': 'query: ', 'input_file': 'data\\processed\\chunks.jsonl', 'input_sha256': '3e0baf1e8796d7f14e983a53664d097dae6463bfad40b68175dc4749d7573e84', 'chunk_count': 1046, 'embedding_dimension': 384, 'artifact_file': 'chunk_embeddings.npz'}


## 4. FAISS vector search

FAISS stores the chunk embeddings in a searchable vector index. This project uses `IndexFlatIP`: an exact inner-product search. Because embeddings are normalized, inner-product ranking is equivalent to cosine-similarity ranking.

Building the index reuses the saved chunk embeddings. It does not generate embeddings again. A search creates only one temporary embedding for the user question, then returns the top five source chunks.

In [7]:
from rag_assistant.retrieval import build_faiss_index, search_chunks
from rag_assistant.config import load_retrieval_settings

faiss_directory = project_root / 'artifacts' / 'faiss'
index_file = faiss_directory / 'chunks.index'
faiss_manifest_path = faiss_directory / 'manifest.json'

current_embeddings_sha256 = hashlib.sha256(embedding_file.read_bytes()).hexdigest()
existing_faiss_manifest = (
    json.loads(faiss_manifest_path.read_text(encoding='utf-8'))
    if faiss_manifest_path.exists()
    else {}
)
if index_file.exists() and existing_faiss_manifest.get('embedding_artifact_sha256') == current_embeddings_sha256:
    faiss_manifest = existing_faiss_manifest
    print('Using existing local FAISS index.')
else:
    faiss_manifest = build_faiss_index(embedding_file, faiss_directory)

print(faiss_manifest)

Using existing local FAISS index.
{'index_type': 'IndexFlatIP', 'embedding_dimension': 384, 'vector_count': 1046, 'embedding_artifact': 'artifacts\\embeddings\\chunk_embeddings.npz', 'embedding_artifact_sha256': '9522dfc973a46a39f33809b21540d5382fc9299bb34a461da2442807ea41b5a7', 'index_file': 'chunks.index', 'chunk_ids': ['skl-pitfalls:001:chunk:001', 'skl-pitfalls:002:chunk:001', 'skl-pitfalls:002:chunk:002', 'skl-pitfalls:003:chunk:001', 'skl-pitfalls:004:chunk:001', 'skl-pitfalls:005:chunk:001', 'skl-pitfalls:005:chunk:002', 'skl-pitfalls:005:chunk:003', 'skl-pitfalls:006:chunk:001', 'skl-pitfalls:007:chunk:001', 'skl-pitfalls:008:chunk:001', 'skl-pitfalls:008:chunk:002', 'skl-pitfalls:009:chunk:001', 'skl-pitfalls:010:chunk:001', 'skl-pitfalls:011:chunk:001', 'skl-pitfalls:011:chunk:002', 'skl-pitfalls:011:chunk:003', 'skl-pitfalls:012:chunk:001', 'skl-pitfalls:012:chunk:002', 'skl-pitfalls:013:chunk:001', 'skl-pitfalls:014:chunk:001', 'skl-pitfalls:014:chunk:002', 'skl-compose:001

In [8]:
from rag_assistant.embeddings import load_embedding_model
from rag_assistant.retrieval import load_retrieval_data
embedding_model = load_embedding_model(embedding_settings)
retrieval_data = load_retrieval_data(chunks_path, embedding_file, index_file)
question = 'Comment traiter les valeurs manquantes ?'
retrieval_settings = load_retrieval_settings(project_root / 'config' / 'settings.toml')

results = search_chunks(
    question,
    chunks_path,
    embedding_file,
    index_file,
    embedding_settings,
    retrieval_settings,
    embedding_model=embedding_model, retrieval_data=retrieval_data,
)

for rank, result in enumerate(results, start=1):
    print(f"{rank}. score={result['score']:.4f}")
    print(f"   heading={' > '.join(result['section_path'])}")
    print(f"   source={result['source_url']}")
    print(f"   preview={' '.join(result['text'].split())[:350]}\n")

C:\Users\cheml\Desktop\Formation AgentOps\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. score=0.8202
   heading=8.4. Imputation of missing values
   source=https://scikit-learn.org/stable/modules/impute.html
   preview=For various reasons, many real world datasets contain missing values, often encoded as blanks, NaNs or other placeholders. Such datasets however are incompatible with scikit-learn estimators which assume that all values in an array are numerical, and that all have and hold meaning. A basic strategy to use incomplete datasets is to discard entire ro

2. score=0.8120
   heading=IO tools (text, CSV, HDF5, …) > CSV & text files > Parsing options > Error handling
   source=https://pandas.pydata.org/docs/user_guide/io.html
   preview=Specifies what to do upon encountering a bad line (a line with too many fields). Allowed values are : - ‘error’, raise a ParserError when a bad line is encountered. - ‘warn’, print a warning when a bad line is encountered and skip that line. - ‘skip’, skip bad lines without raising or warning when they are encountered.

3. score=0

## 5. Grounded generation with Ollama

Retrieval supplies evidence; it does not write the final answer. The local Ollama model receives only the selected passages, the user question and the reusable prompt from `prompts/answer.txt`. The answer must cite the numbered passages and abstain when the retrieved evidence is insufficient.

The first generation cell requires the Ollama application and the model configured in `config/settings.toml` (currently `qwen2.5:1.5b`).

In [9]:
from rag_assistant.config import load_generation_settings
from rag_assistant.generation import generate_answer, translate_retrieval_query

generation_settings = load_generation_settings(project_root / 'config' / 'settings.toml')
prompt_path = project_root / 'prompts' / 'answer.txt'

retrieval_query = translate_retrieval_query(question, generation_settings)
retrieved = search_chunks(
    retrieval_query, chunks_path, embedding_file, index_file,
    embedding_settings, retrieval_settings,
    embedding_model=embedding_model, retrieval_data=retrieval_data,
)
answer = generate_answer(
    question, retrieved, generation_settings, prompt_path, retrieval_query
)
print(f'Retrieval query: {retrieval_query}')
print(answer)

Retrieval query: Handle missing values?
Pour gérer les valeurs manquantes dans les données, vous pouvez utiliser les méthodes suivantes :

1. Utilisez la variable `na_values` pour spécifier les valeurs à considérer comme manquantes. Par défaut, les valeurs manquantes sont représentées par `np.nan`.

2. Utilisez la variable `keep_default_na` pour contrôler si les valeurs par défaut doivent être conservées. Par défaut, les valeurs par défaut sont conservées lors de l'importation.

3. Utilisez la variable `na_values` avec `keep_default_na=False` pour compléter les valeurs manquantes avec les valeurs par défaut.

4. Utilisez la méthode `isna()` pour vérifier si une valeur est manquante.

5. Utilisez la méthode `fillna()` pour remplacer les valeurs manquantes par une valeur spécifique.

6. Utilisez la méthode `dropna()` pour supprimer les lignes ou colonnes contenant des valeurs manquantes.

7. Utilisez la méthode `interpolate()` pour remplacer les valeurs manquantes par des valeurs approxi

## 6. Several questions

A single example can hide fragile behavior. The following small set exercises missing values, pipelines, cross-validation, a French question and a question outside the selected corpus. The answers are printed with their retrieved source labels so that the result can be reviewed manually.

In [10]:
questions = [
    'How can missing values be imputed in scikit-learn?',
    'What is the purpose of a Pipeline?',
    'Comment évaluer les performances d’un estimateur ?',
    'How should I deploy this project to Kubernetes?',
]

for question_item in questions:
    print(f'Running: {question_item}', flush=True)
    query_item = translate_retrieval_query(question_item, generation_settings)
    retrieved_item = search_chunks(
        query_item, chunks_path, embedding_file, index_file,
        embedding_settings, retrieval_settings,
        embedding_model=embedding_model, retrieval_data=retrieval_data,
    )
    answer_item = generate_answer(
        question_item, retrieved_item, generation_settings, prompt_path, query_item
    )
    print(f'\n### {question_item}\n{answer_item}')


Running: How can missing values be imputed in scikit-learn?



### How can missing values be imputed in scikit-learn?
To impute missing values in scikit-learn, you can use the KNNImputer class for the k-Nearest Neighbors approach or the SimpleImputer class for basic strategies. The KNNImputer class uses a Euclidean distance metric that supports missing values, while the SimpleImputer class allows for imputing missing values with a constant value or using the statistics of each column. When using the SimpleImputer class, you can specify the missing values encoding, such as np.nan, and the strategy, such as 'mean' or 'median'. The SimpleImputer class also supports sparse matrices and categorical data. When using the keep_empty_features parameter, you can keep the empty features by imputing with a constant value, which is typically zero.

Sources used: [5] [3] [2]
Running: What is the purpose of a Pipeline?



### What is the purpose of a Pipeline?
A Pipeline serves multiple purposes in scikit-learn. It chains multiple estimators together, allowing you to treat them as a single entity. This is particularly useful when there is a fixed sequence of steps in processing data, such as feature selection, normalization, and classification. By calling `fit` and `predict` once on your data, you can fit a whole sequence of estimators. Additionally, you can grid search over parameters of all estimators in the pipeline at once, ensuring that the same samples are used to train the transformers and predictors. Pipelines help avoid leaking statistics from your test data into the trained model in cross-validation, by ensuring that the same samples are used to train the transformers and predictors. All estimators in a pipeline, except the last one, must be transformers (i.e., must have a `transform` method). The last estimator may be any type (transformer, classifier, etc.). Calling `fit` on the pipeline is


### Comment évaluer les performances d’un estimateur ?
Cross-validation est un outil puissant pour évaluer la performance d'un estimateur. Il permet d'estimer la plage de ses scores attendus. Cependant, un estimateur bien formé sur un grand nombre de données sans structure peut parfois obtenir des performances supérieures que ce n'est pas le cas. Cette performance peut être due à des erreurs de chance. 

La méthode de test de permutation fournit des informations sur la présence d'une dépendance réelle entre les caractéristiques et les cibles. Elle peut aider à évaluer la performance de l'estimateur. Cependant, cette méthode peut produire des p-valeurs faibles même si la structure de données est faible car les datasets permutés ne contiennent aucune structure. 

Il est important de noter que cette méthode produit des p-valeurs faibles même si la structure de données est faible car les datasets permutés ne contiennent aucune structure. C'est pourquoi cette méthode ne peut montrer que la


### How should I deploy this project to Kubernetes?
Deploying this project to Kubernetes involves several steps. First, ensure your model is persisted using one of the methods mentioned: ONNX, skops.io, pickle, or joblib. For ONNX, you need an ONNX runtime and an environment with the necessary dependencies installed. For skops.io, pickle, or joblib, you need a Python environment with the appropriate dependencies installed.

Once the model is persisted, you can deploy it to Kubernetes by creating a Kubernetes deployment or a stateful set. This involves defining the container image, environment variables, and any additional configurations required for your model. Ensure that the Kubernetes cluster has the necessary resources to run the model, including the required Python version and any dependencies.

Remember to monitor the deployment and ensure that the model is accessible and functioning correctly in the Kubernetes environment.

Sources used: [4] [5] [3]


## 7. Evaluation

The versioned evaluation set contains 15 questions: English and French queries, single-source and multi-source questions, and two questions deliberately outside the corpus. The evaluator checks whether an expected document was retrieved, whether citations refer to retrieved passages, and whether out-of-corpus questions trigger abstention. These automatic checks are useful regression signals, but they do not establish factual correctness.

In [11]:
from rag_assistant.embeddings import load_embedding_model
from rag_assistant.evaluation import evaluate_questions
from rag_assistant.retrieval import load_retrieval_data

evaluation_output = project_root / 'artifacts' / 'evaluation'
evaluation_results = evaluate_questions(
    questions_file=project_root / 'data' / 'evaluation' / 'questions.jsonl',
    output_directory=evaluation_output,
    chunks_file=chunks_path,
    embedding_file=embedding_file,
    index_file=index_file,
    prompt_file=prompt_path,
    embedding_settings=embedding_settings,
    retrieval_settings=retrieval_settings,
    generation_settings=generation_settings,
    embedding_model=embedding_model,
    retrieval_data=retrieval_data,
    progress_callback=lambda position, total, identifier: print(f'{position}/{total}: {identifier}', flush=True),
)

retrieval_passes = sum(item['retrieval_pass'] for item in evaluation_results)
answer_passes = sum(item['answer_check_pass'] for item in evaluation_results)
print(f'Retrieval checks passed: {retrieval_passes}/{len(evaluation_results)}')
print(f'Answer format checks passed: {answer_passes}/{len(evaluation_results)}')
print(f'Report: {evaluation_output / "evaluation_report.md"}')
for item in evaluation_results:
    print(f"\n{item['id']} | {item['automatic_status']} | {item['duration_seconds']}s")
    print(item['question'])
    print(item['answer'] or item['error'])
assert not any(item['error'] for item in evaluation_results), 'Evaluation contains execution errors; inspect results above.'

1/15: q01


2/15: q02


3/15: q03


4/15: q04


5/15: q05


...

12/15: q12


13/15: q13


14/15: q14


15/15: q15


Retrieval checks passed: 15/15
Answer format checks passed: 13/15
Report: C:\Users\cheml\Desktop\Formation AgentOps\RAG\artifacts\evaluation\evaluation_report.md

q01 | PASS | 112.47s
How should a Pipeline be designed to prevent data leakage during preprocessing and cross-validation?
To prevent data leakage during preprocessing and cross-validation, a Pipeline should be designed as follows:

1. Always split the data into train and test subsets first, particularly before any preprocessing steps.
2. Never include test data when using the fit and fit_transform methods. Use the fit_transform method on the train subset and transform on the test subset.
3. The scikit-learn pipeline ensures that the appropriate method is performed on the correct data subset. This is ideal for use in cross-validation and hyper-parameter tuning functions.
4. The Pipeline should be designed to ensure that the preprocessing transformations are only learnt from the training data. This prevents information leakage 

## 8. Evaluation and model limits

This notebook demonstrates a transparent educational baseline, not a production-grade assistant. The allowlist covers only 25 documentation pages, so correct answers outside that corpus cannot be expected. Fixed-window chunks can split a concept across boundaries, and dense retrieval may miss exact terminology.

The small local generation model can confuse related concepts, omit constraints or produce plausible wording that is not fully supported by its passages. Citation labels show which passages were supplied, but they do not prove that every sentence is entailed by those passages. The automatic evaluation therefore needs manual review, especially for factual accuracy, completeness, language quality and safe abstention.

The saved run completed all 15 evaluation questions without execution errors. All 13 answerable questions retrieved an expected document; the evaluator reports 15/15 retrieval checks because it automatically passes the two questions with no expected documents. Answer-format checks passed for 13/15 questions, but both out-of-corpus questions failed to abstain. The first French example and evaluation question q01 also reached the output limit with repetitive text. A PASS label is not a factual-quality score.

Useful follow-up experiments include tokenizer-aware or structure-aware chunking, hybrid lexical plus dense retrieval, reranking, a stronger local model, and a larger human-reviewed evaluation set.